In [ ]:
import torch
torch.save(model.state_dict(),'/content/drive/MyDrive/archive/memotion_multimodal.pth')
print('saved')


saved


In [ ]:
import torch
torch.save(model.state_dict(),'/content/drive/MyDrive/archive/memotion_multimodal.pth')
print('saved')


saved


# Memotion Multimodal Sarcasm Detection (CLIP + RoBERTa)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [ ]:
DATASET_PATH='/content/drive/MyDrive/archive/memotion_dataset_7k'
import os, pandas as pd
print(os.listdir(DATASET_PATH))
df=pd.read_csv(f'{DATASET_PATH}/labels.csv')
print(df.head())


['reference.csv', 'labels.csv', 'labels.xlsx', 'labels_pd_pickle', 'reference.xlsx', 'reference_df_pickle', 'images']
   Unnamed: 0    image_name  \
0           0   image_1.jpg   
1           1  image_2.jpeg   
2           2   image_3.JPG   
3           3   image_4.png   
4           4   image_5.png   

                                            text_ocr  \
0  LOOK THERE MY FRIEND LIGHTYEAR NOW ALL SOHALIK...   
1  The best of #10 YearChallenge! Completed in le...   
2  Sam Thorne @Strippin ( Follow Follow Saw every...   
3              10 Year Challenge - Sweet Dee Edition   
4  10 YEAR CHALLENGE WITH NO FILTER 47 Hilarious ...   

                                      text_corrected      humour  \
0  LOOK THERE MY FRIEND LIGHTYEAR NOW ALL SOHALIK...   hilarious   
1  The best of #10 YearChallenge! Completed in le...   not_funny   
2  Sam Thorne @Strippin ( Follow Follow Saw every...  very_funny   
3              10 Year Challenge - Sweet Dee Edition  very_funny   
4  10 YEAR CHALLEN

In [ ]:
!pip -q install transformers accelerate torch torchvision pillow scikit-learn tqdm


In [ ]:
import pandas as pd
df=df[['image_name','text_corrected','sarcasm']].dropna()
df['label']=df['sarcasm'].apply(lambda x: 0 if str(x).lower()=='not_sarcastic' else 1)
from sklearn.model_selection import train_test_split
train_df,test_df=train_test_split(df,test_size=0.2,random_state=42,stratify=df['label'])


In [ ]:
import torch
from transformers import CLIPProcessor,CLIPModel,RobertaTokenizer,RobertaModel
device=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
clip_processor=CLIPProcessor.from_pretrained('openai/clip-vit-base-patch32')
clip_model=CLIPModel.from_pretrained('openai/clip-vit-base-patch32').to(device)
tokenizer=RobertaTokenizer.from_pretrained('roberta-base')
roberta_model=RobertaModel.from_pretrained('roberta-base').to(device)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


preprocessor_config.json:   0%|          | 0.00/316 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/4.19k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/592 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/862k [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaModel LOAD REPORT from: roberta-base
Key                       | Status     | 
--------------------------+------------+-
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
pooler.dense.bias         | MISSING    | 
pooler.dense.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [ ]:
import torch
from transformers import CLIPProcessor,CLIPModel,RobertaTokenizer,RobertaModel

device=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
clip_processor=CLIPProcessor.from_pretrained('openai/clip-vit-base-patch32')
clip_model=CLIPModel.from_pretrained('openai/clip-vit-base-patch32').to(device)
tokenizer=RobertaTokenizer.from_pretrained('roberta-base')
roberta_model=RobertaModel.from_pretrained('roberta-base').to(device)

for p in clip_model.parameters(): p.requires_grad=False
for p in roberta_model.parameters(): p.requires_grad=False

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaModel LOAD REPORT from: roberta-base
Key                       | Status     | 
--------------------------+------------+-
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
pooler.dense.bias         | MISSING    | 
pooler.dense.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [ ]:
from torch.utils.data import Dataset,DataLoader
from PIL import Image
import os, torch
IMAGE_FOLDER=f'{DATASET_PATH}/images'
class MemotionDataset(Dataset):
    def __init__(self,df): self.df=df.reset_index(drop=True)
    def __len__(self): return len(self.df)
    def __getitem__(self,idx):
        row=self.df.iloc[idx]
        img=Image.open(os.path.join(IMAGE_FOLDER,row['image_name'])).convert('RGB')
        txt=tokenizer(str(row['text_corrected']),max_length=128,truncation=True,padding='max_length',return_tensors='pt')
        im=clip_processor(images=img,return_tensors='pt')
        return {'input_ids':txt['input_ids'].squeeze(),'attention_mask':txt['attention_mask'].squeeze(),'pixel_values':im['pixel_values'].squeeze(),'label':torch.tensor(int(row['label']))}
train_loader=DataLoader(MemotionDataset(train_df),batch_size=16,shuffle=True)
test_loader=DataLoader(MemotionDataset(test_df),batch_size=16)


In [ ]:
import torch.nn as nn
class Model(nn.Module):
    def __init__(self):
        super().__init__()
        self.classifier=nn.Sequential(nn.Linear(1280,512),nn.ReLU(),nn.Dropout(0.3),nn.Linear(512,128),nn.ReLU(),nn.Linear(128,2))
    def forward(self,t,i):
        return self.classifier(torch.cat([t,i],dim=1))
model=Model().to(device)
criterion=nn.CrossEntropyLoss()
optimizer=torch.optim.Adam(model.parameters(),lr=1e-4)


In [ ]:
import torch
from tqdm import tqdm
import pandas as pd
from sklearn.model_selection import train_test_split
import os
import torch.nn as nn

# Define DATASET_PATH and load df (copied from bjtCu-Amg9Gw)
DATASET_PATH='/content/drive/MyDrive/archive/memotion_dataset_7k'
df=pd.read_csv(f'{DATASET_PATH}/labels.csv')
IMAGE_FOLDER=f'{DATASET_PATH}/images'

# Re-process df and create train_df, test_df (copied from HtL_c-Nag9Gy)
df_processed = df[['image_name','text_corrected','sarcasm']].dropna()
df_processed['label']=df_processed['sarcasm'].apply(lambda x: 0 if str(x).lower()=='not_sarcastic' else 1)
train_df,test_df=train_test_split(df_processed,test_size=0.2,random_state=42,stratify=df_processed['label'])

# Redefine MemotionDataset to handle FileNotFoundError and OSError
class MemotionDataset(torch.utils.data.Dataset):
    def __init__(self, df): self.df = df.reset_index(drop=True)
    def __len__(self): return len(self.df)
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        try:
            img_path = os.path.join(IMAGE_FOLDER, row['image_name'])
            img = Image.open(img_path).convert('RGB')
        except (FileNotFoundError, OSError):
            print(f"Warning: Image file not found or corrupted: {img_path}. Skipping this item.")
            return None # Return None for missing or corrupted images

        txt = tokenizer(str(row['text_corrected']), max_length=128, truncation=True, padding='max_length', return_tensors='pt')
        im = clip_processor(images=img, return_tensors='pt')

        return {'input_ids': txt['input_ids'].squeeze(), 'attention_mask': txt['attention_mask'].squeeze(), 'pixel_values': im['pixel_values'].squeeze(), 'label': torch.tensor(int(row['label']))}

# Define a custom collate_fn to filter out None values
def custom_collate_fn(batch):
    batch = [item for item in batch if item is not None] # Filter out None entries
    if not batch:
        return None # Return None if all items in the batch were invalid/None

    # Assuming all items in batch are dictionaries with the same keys
    # And values are tensors that can be stacked.
    collated_batch = {}
    for key in batch[0].keys():
        collated_batch[key] = torch.stack([d[key] for d in batch])
    return collated_batch

# Re-instantiate DataLoader with the modified dataset and custom collate_fn
train_loader = torch.utils.data.DataLoader(MemotionDataset(train_df), batch_size=16, shuffle=True, collate_fn=custom_collate_fn)
test_loader = torch.utils.data.DataLoader(MemotionDataset(test_df), batch_size=16, collate_fn=custom_collate_fn)

# Define Model, criterion, and optimizer (copied from 6ENHCJrJg9G2)
class Model(nn.Module):
    def __init__(self):
        super().__init__()
        self.classifier=nn.Sequential(nn.Linear(1280,512),nn.ReLU(),nn.Dropout(0.3),nn.Linear(512,128),nn.ReLU(),nn.Linear(128,2))
    def forward(self,t,i):
        return self.classifier(torch.cat([t,i],dim=1))
model=Model().to(device)
criterion=nn.CrossEntropyLoss()
optimizer=torch.optim.Adam(model.parameters(),lr=1e-4)

EPOCHS=5
for epoch in range(EPOCHS):
    model.train(); total_loss=0
    for batch in tqdm(train_loader):
        if batch is None: # Skip if the entire batch was invalid/empty
            continue
        with torch.no_grad():
            t=roberta_model(input_ids=batch['input_ids'].to(device),attention_mask=batch['attention_mask'].to(device)).last_hidden_state[:,0,:]
            # Correctly extract the tensor from the BaseModelOutputWithPooling object
            i=clip_model.get_image_features(pixel_values=batch['pixel_values'].to(device)).pooler_output
        out=model(t,i)
        loss=criterion(out,batch['label'].to(device))
        optimizer.zero_grad(); loss.backward(); optimizer.step(); total_loss+=loss.item()
    print(epoch+1,total_loss)


  1%|          | 2/350 [00:16<47:01,  8.11s/it]/usr/local/lib/python3.12/dist-packages/PIL/Image.py:1047: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(
100%|██████████| 350/350 [39:14<00:00,  6.73s/it]


1 187.47917480766773


100%|██████████| 350/350 [02:05<00:00,  2.79it/s]


2 183.52391037344933


100%|██████████| 350/350 [02:05<00:00,  2.80it/s]


3 181.78967607021332


100%|██████████| 350/350 [02:12<00:00,  2.64it/s]


4 178.58418372273445


100%|██████████| 350/350 [02:10<00:00,  2.68it/s]

5 174.4394625276327


### Counting Missing Images

Now, let's count the total number of images that are missing from the dataset. We'll use the `MemotionDataset` to identify entries where the image file is not found.

In [ ]:
# Count missing images in the training dataset
missing_train_images = 0
train_dataset_for_counting = MemotionDataset(train_df)
for i in tqdm(range(len(train_dataset_for_counting)), desc="Counting missing train images"):
    if train_dataset_for_counting[i] is None:
        missing_train_images += 1

print(f"Number of missing images in the training set: {missing_train_images}")

Counting missing train images: 100%|██████████| 5589/5589 [01:17<00:00, 71.98it/s]

Number of missing images in the training set: 0


In [ ]:
# Count missing images in the testing dataset
missing_test_images = 0
test_dataset_for_counting = MemotionDataset(test_df)
for i in tqdm(range(len(test_dataset_for_counting)), desc="Counting missing test images"):
    if test_dataset_for_counting[i] is None:
        missing_test_images += 1

print(f"Number of missing images in the test set: {missing_test_images}")

total_missing_images = missing_train_images + missing_test_images
print(f"Total number of missing images: {total_missing_images}")

Counting missing test images:  17%|█▋        | 237/1398 [01:34<07:41,  2.52it/s]


KeyboardInterrupt: 

In [ ]:
missing_test_images = 0
test_dataset_for_counting = MemotionDataset(test_df)
for i in tqdm(range(len(test_dataset_for_counting)), desc="Counting missing test images"):
    if test_dataset_for_counting[i] is None:
        missing_test_images += 1

print(f"Number of missing images in the test set: {missing_test_images}")

total_missing_images = missing_train_images + missing_test_images
print(f"Total number of missing images: {total_missing_images}")

Counting missing test images:  43%|████▎     | 600/1398 [02:28<05:20,  2.49it/s]

Counting missing test images: 100%|██████████| 1398/1398 [07:51<00:00,  2.96it/s]

Number of missing images in the test set: 1
Total number of missing images: 1


### Model Evaluation

Now, let's evaluate the trained model on the test dataset to assess its performance using classification metrics such as accuracy, precision, recall, and F1-score.

In [ ]:
from sklearn.metrics import classification_report

model.eval() # Set the model to evaluation mode
all_preds = []
all_labels = []

with torch.no_grad(): # Disable gradient calculations during evaluation
    for batch in tqdm(test_loader, desc="Evaluating model"):
        if batch is None: # Skip if the entire batch was invalid/empty
            continue

        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        pixel_values = batch['pixel_values'].to(device)
        labels = batch['label'].to(device)

        # Extract features
        text_features = roberta_model(input_ids=input_ids, attention_mask=attention_mask).last_hidden_state[:,0,:]
        image_features = clip_model.get_image_features(pixel_values=pixel_values).pooler_output

        # Get predictions
        outputs = model(text_features, image_features)
        _, predicted = torch.max(outputs.data, 1)

        all_preds.extend(predicted.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

# Generate classification report
print("\nClassification Report:")
print(classification_report(all_labels, all_preds, target_names=['not_sarcastic', 'sarcastic']))

Evaluating model:  42%|████▏     | 37/88 [00:13<00:18,  2.75it/s]

Evaluating model: 100%|██████████| 88/88 [00:34<00:00,  2.54it/s]


Classification Report:
               precision    recall  f1-score   support

not_sarcastic       0.22      0.01      0.01       309
    sarcastic       0.78      0.99      0.87      1088

     accuracy                           0.78      1397
    macro avg       0.50      0.50      0.44      1397
 weighted avg       0.66      0.78      0.68      1397



### Predict on new images

Now, let's create a function to use our trained model to predict sarcasm on new images and text. You'll need to provide the paths to your uploaded images and the associated text.

In [ ]:
from PIL import Image
import numpy as np

def predict_sarcasm(image_paths, texts):
    model.eval() # Set model to evaluation mode
    predictions = []

    for img_path, text_input in zip(image_paths, texts):
        try:
            # Process image
            img = Image.open(img_path).convert('RGB')
            pixel_values = clip_processor(images=img, return_tensors='pt').pixel_values.to(device)

            # Process text
            text_encoding = tokenizer(text_input, max_length=128, truncation=True, padding='max_length', return_tensors='pt')
            input_ids = text_encoding['input_ids'].to(device)
            attention_mask = text_encoding['attention_mask'].to(device)

            with torch.no_grad():
                text_features = roberta_model(input_ids=input_ids, attention_mask=attention_mask).last_hidden_state[:, 0, :]
                image_features = clip_model.get_image_features(pixel_values=pixel_values).pooler_output

                outputs = model(text_features, image_features)
                _, predicted = torch.max(outputs.data, 1)
                predictions.append('sarcastic' if predicted.item() == 1 else 'not_sarcastic')

        except FileNotFoundError:
            predictions.append(f"Error: Image not found at {img_path}")
        except Exception as e:
            predictions.append(f"Error processing {img_path} with text '{text_input}': {e}")

    return predictions

#### Example Usage

To use the `predict_sarcasm` function:

1.  **Upload your images** to your Colab environment (e.g., to a folder named `uploaded_images` in your root directory).
2.  **Provide a list of paths** to these images.
3.  **Provide a list of corresponding texts** that go with each image.

Here's an example assuming you've uploaded `my_image1.jpg` and `my_image2.png` to `/content/uploaded_images/`:

In [ ]:
# Example: Replace these with your actual image paths and texts
custom_image_paths = [
    '/content/yyih6qys3b071.gif',
    '/content/yzdxxciyrom41.jpg',
    '/content/yzyo0zym25d61.jpg',
    '/content/z0gc9rwi4lm31.jpg',
    '/content/z0j4h65dns461.jpg',
    '/content/z24pktp1unx51.jpg',
    '/content/z2b6x1xxde491.gif',
    '/content/z93qe7happ771',
    '/content/z9oh7ligb0i31.jpg',
    '/content/zac7gguzgvp51.jpg',
    '/content/zbk235yn8zj61.png',
    '/content/zc36e4egmun41.png'
]
custom_texts = [
    "A meme with an image.",
    "A picture that could be sarcastic.",
    "A humorous image.",
    "Is this sarcastic?",
    "Some text combined with an image.",
    "A funny image with text.",
    "Could this meme be sarcastic?",
    "An image to be analyzed for sarcasm.",
    "A picture with a caption.",
    "A visual joke.",
    "An image with embedded text.",
    "Evaluate this image for sarcasm."
]

# Make predictions
results = predict_sarcasm(custom_image_paths, custom_texts)

# Print the results
for i, (img_path, text, prediction) in enumerate(zip(custom_image_paths, custom_texts, results)):
    print(f"\nPrediction for Image {i+1} ({img_path}) with text '{text}': {prediction}")


Prediction for Image 1 (/content/yyih6qys3b071.gif) with text 'A meme with an image.': sarcastic

Prediction for Image 2 (/content/yzdxxciyrom41.jpg) with text 'A picture that could be sarcastic.': sarcastic

Prediction for Image 3 (/content/yzyo0zym25d61.jpg) with text 'A humorous image.': sarcastic

Prediction for Image 4 (/content/z0gc9rwi4lm31.jpg) with text 'Is this sarcastic?': sarcastic

Prediction for Image 5 (/content/z0j4h65dns461.jpg) with text 'Some text combined with an image.': sarcastic

Prediction for Image 6 (/content/z24pktp1unx51.jpg) with text 'A funny image with text.': sarcastic

Prediction for Image 7 (/content/z2b6x1xxde491.gif) with text 'Could this meme be sarcastic?': sarcastic

Prediction for Image 8 (/content/z93qe7happ771) with text 'An image to be analyzed for sarcasm.': Error processing /content/z93qe7happ771 with text 'An image to be analyzed for sarcasm.': cannot identify image file '/content/z93qe7happ771'

Prediction for Image 9 (/content/z9oh7ligb0

### Analysis of Custom Image Predictions

Let's organize the prediction results for your custom images into a DataFrame and summarize them.

In [ ]:
import pandas as pd

# Create a DataFrame from the prediction results
prediction_df = pd.DataFrame({
    'image_path': custom_image_paths,
    'text_input': custom_texts,
    'prediction': results
})

# Display the DataFrame
display(prediction_df)

# Clean predictions for counting: ignore error messages
clean_predictions = [pred for pred in results if pred in ['sarcastic', 'not_sarcastic']]

# Count the number of 'sarcastic' and 'not_sarcastic' predictions
sarcasm_counts = pd.Series(clean_predictions).value_counts()

print('\nPrediction Summary:')
print(sarcasm_counts)

# Identify which image caused the error
error_entries = prediction_df[prediction_df['prediction'].str.contains('Error')]
if not error_entries.empty:
    print('\nImages with Errors:')
    display(error_entries)


,image_path,text_input,prediction
0,/content/yyih6qys3b071.gif,A meme with an image.,sarcastic
1,/content/yzdxxciyrom41.jpg,A picture that could be sarcastic.,sarcastic
2,/content/yzyo0zym25d61.jpg,A humorous image.,sarcastic
3,/content/z0gc9rwi4lm31.jpg,Is this sarcastic?,sarcastic
4,/content/z0j4h65dns461.jpg,Some text combined with an image.,sarcastic
5,/content/z24pktp1unx51.jpg,A funny image with text.,sarcastic
6,/content/z2b6x1xxde491.gif,Could this meme be sarcastic?,sarcastic
7,/content/z93qe7happ771,An image to be analyzed for sarcasm.,Error processing /content/z93qe7happ771 with t...
8,/content/z9oh7ligb0i31.jpg,A picture with a caption.,sarcastic
9,/content/zac7gguzgvp51.jpg,A visual joke.,sarcastic



Prediction Summary:
sarcastic    11
Name: count, dtype: int64

Images with Errors:


,image_path,text_input,prediction
7,/content/z93qe7happ771,An image to be analyzed for sarcasm.,Error processing /content/z93qe7happ771 with t...


In [ ]:
import torch
torch.save(model.state_dict(),'/content/drive/MyDrive/archive/memotion_multimodal.pth')
print('saved')


saved
